# Experiment 9a: dual-geometry transport (embedding-proximity edges)

See `experiments/experiment_9a_design.md`.

Repair the transport wall by **adding missing paths**: a minority channel of
**semantic-proximity edges** (mutual-kNN on embeddings, hub-safe, throttled
conductance `w_sem`, edge type `SemanticProximity`). Two levers now:
- 8a's *bypass* (`lambda*sim` additive term) — kept only in the `*r` arms;
- 9a's *repair* (semantic edges) — the new variable.

**Ablation ladder** (isolate the edge contribution honestly):
- `G0a` 8a-best score, NO edges (realistic reference)
- `G0b` same but `lambda=0` (NO bypass, NO edges) -- pure transport
- `G1..G3` = G0b + edges at rising `w_sem` (edges on transport ALONE)
- `G1r..G3r` = same edges at 8a-best `lambda` (realistic combined)

Metric: **median rank** of best manifold node (robust to the rank-5000 tails 8a
showed), plus pairwise beat-MAX-distractor. Plus two danger checks: hub-reflood
and RAG-collapse. Grounded by 8a's Q2 (`document -> passport` transport was ALL
zeros -> no path -> exactly what edges should fix).


In [1]:
%load_ext autoreload
%autoreload 2
import sys, time
from pathlib import Path
from collections import defaultdict
import numpy as np
SRC = Path.cwd() / "src"
if not (SRC / "memory").exists():
    SRC = Path.cwd().parent
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
from embryonic import conceptnet_source
from embryonic.loader import EmbryonicGraphLoader
from embryonic.numberbatch import Numberbatch
print("src:", SRC)

src: c:\Users\yaros\BlackSabbitch_repo\RIN\src


In [2]:
SOURCE_DIR = Path.cwd() / "source"
if not SOURCE_DIR.exists():
    SOURCE_DIR = Path.cwd().parent.parent / "source"
CONCEPTNET_DUMP = SOURCE_DIR / "conceptnet-assertions-5.7.0.csv.gz"
NUMBERBATCH_FILE = SOURCE_DIR / "numberbatch-en-19.08.txt.gz"
LANGUAGE = "en"

# 8a seeds + synonym-bridge clusters (for the P1 co-activation probe)
SEED_TERMS = [
    "coffee", "stimulant", "caffeine", "beverage", "hot beverage", "energy", "tea", "espresso", "drink coffee",
    "check in", "document", "passport", "boarding pass", "airport", "suitcase", "identity",
    "laptop", "battery", "electricity", "charger", "power", "power adapter", "office", "work",
    "wake up", "tired", "morning", "sleep", "apartment", "hotel", "neighbor", "rent", "home",
    "trip", "travel", "vacation", "pack", "flight",
    # synonym / morphology clusters for the bridge probe:
    "cat", "kitty", "feline", "kitten", "photo", "photograph", "picture", "image",
    "car", "automobile", "vehicle", "couch", "sofa", "garbage", "trash", "buy", "purchase",
]
RADIUS = 2
LIMIT_PER_NODE = 40
MAX_NODES = 12000

# scoring / diffusion
ETA = 0.25
DECAY = 0.8
MIN_PROP_FRAC = 0.01
MAX_STEPS = 20
LAMBDA_MAIN = 1.0          # 8a-best additive-sim strength (used only in *a and *r arms)
# semantic edges
K_SEM = 8                  # kNN before mutual filter
SIM_FLOOR = 0.40           # min cosine to even consider a semantic edge
W_SEMS = [0.0, 0.1, 0.3, 1.0]   # semantic-channel conductance sweep (0.0 == edges off)

In [3]:
t0 = time.time()
adjacency = conceptnet_source.build_language_adjacency(str(CONCEPTNET_DUMP), LANGUAGE)
print(f"adjacency: {len(adjacency):,} concepts in {time.time()-t0:.0f}s")
t0 = time.time()
numberbatch = Numberbatch()
numberbatch.load(str(NUMBERBATCH_FILE))
print(f"numberbatch loaded in {time.time()-t0:.0f}s")

adjacency: 1,787,373 concepts in 169s
numberbatch loaded in 81s


In [4]:
loader = EmbryonicGraphLoader(
    entries=SEED_TERMS, radius=RADIUS, language=LANGUAGE,
    conceptnet_backend="local_dump", conceptnet_adjacency=adjacency,
    limit_per_node=LIMIT_PER_NODE, max_nodes=MAX_NODES,
    include_atomic=False, numberbatch=numberbatch,
)
t0 = time.time()
G = loader.load()
for node in G.nodes.values():
    if node.embedding is None and node.provenance and isinstance(node.provenance[0], str) and node.provenance[0].startswith("/c/"):
        v = numberbatch.vector_for(node.provenance[0])
        if v is not None:
            node.embedding = v
NODE_IDS = list(G.node_ids())
emb_ids = [n for n in NODE_IDS if G.nodes[n].embedding is not None]
M = np.array([np.asarray(G.nodes[n].embedding, dtype=np.float32) for n in emb_ids])
Mn = M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-12)
emb_pos = {nid: i for i, nid in enumerate(emb_ids)}
print(f"graph: {len(G.nodes)} nodes, embedded {len(emb_ids)} in {time.time()-t0:.1f}s")

graph: 12000 nodes, embedded 6926 in 0.3s


In [5]:
# indices + resolution (same as 8a)
uri2id, content2id = {}, {}
for nid, node in G.nodes.items():
    if node.provenance and isinstance(node.provenance[0], str):
        uri2id.setdefault(node.provenance[0], nid)
    if node.content:
        content2id.setdefault(node.content.strip().lower(), nid)

def resolve(term):
    uri = f"/c/{LANGUAGE}/" + term.strip().lower().replace(" ", "_")
    return uri2id.get(uri) or content2id.get(term.strip().lower())

def resolve_group(terms):
    for t in terms:
        i = resolve(t)
        if i is not None:
            return i, t
    return None, None

In [6]:
# PageRank prior (scipy-free) + direct similarity, as in 8a
print("pagerank...")
t0 = time.time()
def compute_pagerank(G, ids, alpha=0.85, iters=100, tol=1e-9):
    idx = {n: i for i, n in enumerate(ids)}
    N = len(ids)
    out = [G.out_edges(n) for n in ids]
    wsum = [sum(max(e.weight, 1e-12) for e in oe) for oe in out]
    pr = np.full(N, 1.0 / N); tele = (1.0 - alpha) / N
    for _ in range(iters):
        new = np.full(N, tele); dang = 0.0
        for i, oe in enumerate(out):
            if wsum[i] <= 0:
                dang += pr[i]; continue
            c = alpha * pr[i] / wsum[i]
            for e in oe:
                j = idx.get(e.dst)
                if j is not None:
                    new[j] += c * max(e.weight, 1e-12)
        new += alpha * dang / N
        if np.abs(new - pr).sum() < tol:
            pr = new; break
        pr = new
    return {ids[i]: max(float(pr[i]), 1e-12) for i in range(N)}
PR = compute_pagerank(G, NODE_IDS)
def pr_pow(nid, eta=ETA):
    return PR[nid] ** eta
print(f"  done in {time.time()-t0:.1f}s")

def direct_sim(query_vecs):
    sims = {n: 0.0 for n in NODE_IDS}
    if not query_vecs:
        return sims
    Q = np.array(query_vecs, dtype=np.float32)
    Q = Q / (np.linalg.norm(Q, axis=1, keepdims=True) + 1e-12)
    best = (Mn @ Q.T).max(axis=1)
    for i, nid in enumerate(emb_ids):
        sims[nid] = float(max(0.0, best[i]))
    return sims

pagerank...
  done in 2.1s


## Semantic-proximity edges: mutual-kNN, hub-safe

Edge kept only if `v in kNN(u)` AND `u in kNN(v)` (mutual -> kills hub edges),
above `SIM_FLOOR`. This is the `SemanticProximity` relation type; its conductance
in diffusion is `w_sem` (a throttled minority channel).

In [7]:
def build_semantic_edges(emb_ids, Mn, k=K_SEM, sim_floor=SIM_FLOOR, block=1024):
    N = len(emb_ids)
    knn = [dict() for _ in range(N)]  # i -> {j: sim}
    for s in range(0, N, block):
        e = min(s + block, N)
        B = Mn[s:e] @ Mn.T  # (b, N)
        for bi in range(e - s):
            i = s + bi
            row = B[bi].copy(); row[i] = -2.0  # exclude self
            idx = np.argpartition(-row, min(k, N - 1))[:k]
            for j in idx:
                j = int(j)
                if row[j] >= sim_floor:
                    knn[i][j] = float(row[j])
    sem_adj = defaultdict(list)  # node_id -> [(node_id, sim)]
    n_edges = 0
    for i in range(N):
        for j, sij in knn[i].items():
            if i in knn[j]:  # mutual
                sem_adj[emb_ids[i]].append((emb_ids[j], sij)); n_edges += 1
    degs = [len(v) for v in sem_adj.values()]
    print(f"semantic edges: {n_edges} directed ({n_edges//2} undirected), "
          f"nodes with >=1: {len(sem_adj)}, max degree: {max(degs) if degs else 0}")
    return sem_adj

t0 = time.time()
SEM_ADJ = build_semantic_edges(emb_ids, Mn)
print(f"built in {time.time()-t0:.1f}s")

semantic edges: 27978 directed (13989 undirected), nodes with >=1: 6307, max degree: 8
built in 0.9s


In [8]:
RELATEDTO = {"relatedto", "related_to"}

def diffuse_dual(G, sources, sem_adj, w_sem=0.0, decay=DECAY, min_prop_frac=MIN_PROP_FRAC,
                 max_steps=MAX_STEPS, restart=0.0, rel=None, c_relatedto=1.0):
    """diffusion over graph edges UNION semantic edges (conductance w_sem).
    w_sem=0 -> semantic channel off -> identical to graph-only transport."""
    current = dict(sources); peak = dict(current)
    floor = min_prop_frac * (max(sources.values()) if sources else 0.0)
    for _ in range(max_steps):
        active = {n: a for n, a in current.items() if a >= floor}
        if not active:
            break
        nxt = defaultdict(float)
        for n, a in active.items():
            tgt, ws = [], []
            for e in G.out_edges(n):
                w = e.weight * (c_relatedto if (e.type or "").lower() in RELATEDTO else 1.0)
                tgt.append(e.dst); ws.append(w)
            if w_sem > 0:
                for nb, sw in sem_adj.get(n, ()): 
                    tgt.append(nb); ws.append(sw * w_sem)
            if rel is not None:
                ws = [w * rel.get(t, 1.0) for w, t in zip(ws, tgt)]
            tot = sum(ws)
            if tot <= 0:
                continue
            for t, w in zip(tgt, ws):
                nxt[t] += decay * a * (w / tot)
        if restart > 0:
            for sN, amt in sources.items():
                nxt[sN] += restart * amt
        current = dict(nxt)
        for n, a in current.items():
            if a > peak.get(n, 0.0):
                peak[n] = a
    return peak

## Methods (ablation ladder) and query suite

In [9]:
# method -> (w_sem, lambda)
METHODS = {
    "G0a": (0.0, LAMBDA_MAIN),   # 8a-best score, no edges (realistic reference)
    "G0b": (0.0, 0.0),           # pure transport, no bypass, no edges
    "G1":  (W_SEMS[1], 0.0),     # edges on transport ALONE
    "G2":  (W_SEMS[2], 0.0),
    "G3":  (W_SEMS[3], 0.0),
    "G1r": (W_SEMS[1], LAMBDA_MAIN),  # edges + bypass (realistic combined)
    "G2r": (W_SEMS[2], LAMBDA_MAIN),
    "G3r": (W_SEMS[3], LAMBDA_MAIN),
}
METHOD_ORDER = ["G0b", "G1", "G2", "G3", "G0a", "G1r", "G2r", "G3r"]

def query_setup(groups):
    res = [resolve_group(g) for g in groups]
    anchor_ids = [r[0] for r in res if r[0] is not None]
    src = {a: 100.0 for a in anchor_ids}
    qvecs = [np.asarray(G.nodes[a].embedding, dtype=np.float32)
             for a in anchor_ids if G.nodes[a].embedding is not None]
    return anchor_ids, src, qvecs

def scores(groups, w_sem, lam, sem_adj=SEM_ADJ):
    anchor_ids, src, qvecs = query_setup(groups)
    sim = direct_sim(qvecs)
    t = diffuse_dual(G, src, sem_adj, w_sem=w_sem)
    S = {n: (t.get(n, 0.0) + lam * sim[n]) / pr_pow(n) for n in NODE_IDS}
    return set(anchor_ids), sim, S

# PRIMARY probes: the representation-gap / missing-path cases from 8a
QUERIES = [
    ("primary", "Ppass", [["check in", "checking in"], ["document"]],
        ["passport", "boarding pass"], ["airport", "checking in", "storing information"]),
    ("primary", "Pcaf", [["coffee"], ["stimulant", "energy"]],
        ["caffeine"], ["beverage", "hot beverage", "popular drink"]),
    ("primary", "Pchg", [["laptop"], ["battery", "electricity"]],
        ["charger", "power adapter"], ["office", "power"]),
    ("control", "Capt", [["apartment", "neighbor"], ["rent", "home"]],
        ["apartment", "home"], ["trip"]),
    ("control", "Ctrv", [["trip", "travel"], ["vacation", "flight"]],
        ["trip", "travel"], ["apartment"]),
]
for kind, qid, groups, man, dist in QUERIES:
    a = [resolve_group(g)[1] for g in groups]
    mo = [t for t in man if resolve(t) is not None]
    do = [t for t in dist if resolve(t) is not None]
    print(f"{qid} [{kind}] anchors={a} manifold_ok={mo} distractors_ok={do}")

Ppass [primary] anchors=['check in', 'document'] manifold_ok=['passport', 'boarding pass'] distractors_ok=['airport', 'checking in', 'storing information']
Pcaf [primary] anchors=['coffee', 'stimulant'] manifold_ok=['caffeine'] distractors_ok=['beverage', 'hot beverage', 'popular drink']
Pchg [primary] anchors=['laptop', 'battery'] manifold_ok=['charger', 'power adapter'] distractors_ok=['office', 'power']
Capt [control] anchors=['apartment', 'rent'] manifold_ok=['apartment', 'home'] distractors_ok=['trip']
Ctrv [control] anchors=['trip', 'vacation'] manifold_ok=['trip', 'travel'] distractors_ok=['apartment']


## Main ablation table: median rank + pairwise (beat MAX distractor)

Key question `G0b -> G1..G3`: do edges help **with bypass off**? Then `G1r..G3r`
checks edges compose with bypass. Median rank is robust to the rank-5000 tails.

In [10]:
import statistics

def best_rank(S, src_set, target_ids):
    sc = {n: S[n] for n in NODE_IDS if n not in src_set}
    order = [n for n, _ in sorted(sc.items(), key=lambda kv: -kv[1])]
    pos = {n: r for r, n in enumerate(order, start=1)}
    ranks = [pos[i] for i in target_ids if i in pos]
    return (min(ranks) if ranks else None), sc

def evaluate():
    agg = {m: {"p_rank": [], "p_pair": 0, "p_n": 0, "c_pair": 0, "c_n": 0} for m in METHOD_ORDER}
    for kind, qid, groups, man, dist in QUERIES:
        man_ids0 = [resolve(t) for t in man]
        dist_ids0 = [resolve(t) for t in dist]
        for m in METHOD_ORDER:
            w_sem, lam = METHODS[m]
            src_set, sim, S = scores(groups, w_sem, lam)
            man_ids = [i for i in man_ids0 if i is not None and i not in src_set]
            dist_ids = [i for i in dist_ids0 if i is not None and i not in src_set]
            if not man_ids or not dist_ids:
                continue
            r, sc = best_rank(S, src_set, man_ids)
            dmax = max(sc.get(i, 0.0) for i in dist_ids)
            bman = max(sc.get(i, 0.0) for i in man_ids)
            if kind == "primary":
                agg[m]["p_n"] += 1
                agg[m]["p_pair"] += int(bman > dmax)
                if r:
                    agg[m]["p_rank"].append(r)
            else:
                agg[m]["c_n"] += 1
                agg[m]["c_pair"] += int(bman > dmax)
    return agg

agg = evaluate()
print("method | (w_sem, lam) | p.pairwise | median_rank | control")
for m in METHOD_ORDER:
    a = agg[m]
    med = statistics.median(a["p_rank"]) if a["p_rank"] else float("nan")
    print(f"{m:5s} | {str(METHODS[m]):12s} | {a['p_pair']}/{a['p_n']:<8} | {med:11.1f} | {a['c_pair']}/{a['c_n']}")

method | (w_sem, lam) | p.pairwise | median_rank | control
G0b   | (0.0, 0.0)   | 1/3        |       249.0 | 2/2
G1    | (0.1, 0.0)   | 1/3        |       127.0 | 2/2
G2    | (0.3, 0.0)   | 1/3        |        90.0 | 2/2
G3    | (1.0, 0.0)   | 1/3        |        43.0 | 2/2
G0a   | (0.0, 1.0)   | 1/3        |       111.0 | 2/2
G1r   | (0.1, 1.0)   | 1/3        |        98.0 | 2/2
G2r   | (0.3, 1.0)   | 1/3        |        59.0 | 2/2
G3r   | (1.0, 1.0)   | 0/3        |        42.0 | 2/2


## Probe P1: synonymy/morphology bridge

Auto-pick English pairs that are embedding-near but graph-disconnected (no direct
edge), inject one, and measure the partner's rank with edges off vs on. This is
the cleanest 'do the edges bridge a real gap' test.

In [11]:
CANDIDATE_PAIRS = [("cat", "kitty"), ("cat", "feline"), ("photo", "photograph"),
                   ("photo", "picture"), ("car", "automobile"), ("couch", "sofa"),
                   ("garbage", "trash"), ("buy", "purchase")]

def direct_linked(u, v):
    if any(e.dst == v for e in G.out_edges(u)):
        return True
    if any(e.dst == u for e in G.out_edges(v)):
        return True
    return False

def cos(u, v):
    if u not in emb_pos or v not in emb_pos:
        return None
    return float(Mn[emb_pos[u]] @ Mn[emb_pos[v]])

print("pair                     | cos  | graph-linked | rank(partner) w_sem=0 -> 0.3 -> 1.0")
for a, b in CANDIDATE_PAIRS:
    ia, ib = resolve(a), resolve(b)
    if ia is None or ib is None:
        print(f"{a}/{b}: unresolved"); continue
    c = cos(ia, ib)
    linked = direct_linked(ia, ib)
    ranks = []
    for w in (0.0, 0.3, 1.0):
        peak = diffuse_dual(G, {ia: 100.0}, SEM_ADJ, w_sem=w)
        order = [n for n, _ in sorted(((n, peak.get(n, 0.0)) for n in NODE_IDS if n != ia), key=lambda kv: -kv[1])]
        pos = {n: r for r, n in enumerate(order, start=1)}
        ranks.append(pos.get(ib, None))
    cs = f"{c:.2f}" if c is not None else "  na"
    print(f"{a+'/'+b:24s} | {cs} | {str(linked):12s} | {ranks}")

pair                     | cos  | graph-linked | rank(partner) w_sem=0 -> 0.3 -> 1.0
cat/kitty                | 0.82 | False        | [1161, 285, 278]
cat/feline               | 0.86 | True         | [1, 1, 1]
photo/photograph         | 0.91 | True         | [1, 1, 1]
photo/picture            | 0.77 | True         | [4, 4, 13]
car/automobile           | 0.88 | True         | [9, 10, 7]
couch/sofa               | 0.94 | False        | [1488, 37, 22]
garbage/trash            | 0.94 | True         | [1, 1, 1]
buy/purchase             | 0.87 | True         | [2, 2, 2]


## Danger 1: hub re-flood

As `w_sem` rises, generic distractors must NOT climb. Report best-distractor
median rank across the primary queries per w_sem (higher = safer).

In [12]:
import statistics
print("w_sem | generic-distractor median rank (higher = distractor stays down = safe)")
for w in W_SEMS:
    ranks = []
    for kind, qid, groups, man, dist in QUERIES:
        if kind != "primary":
            continue
        src_set, sim, S = scores(groups, w, 0.0)
        sc = {n: S[n] for n in NODE_IDS if n not in src_set}
        order = [n for n, _ in sorted(sc.items(), key=lambda kv: -kv[1])]
        pos = {n: r for r, n in enumerate(order, start=1)}
        for t in dist:
            i = resolve(t)
            if i is not None and i in pos:
                ranks.append(pos[i])
    med = statistics.median(ranks) if ranks else float("nan")
    print(f"{w:5.2f} | {med:.0f}")

w_sem | generic-distractor median rank (higher = distractor stays down = safe)
 0.00 | 22
 0.10 | 24
 0.30 | 26
 1.00 | 22


## Danger 2: RAG-collapse

As `w_sem` rises the ranking must not degenerate into pure embedding-kNN (graph
relations drowned). Measure top-20 overlap between the dual-transport ranking and
a pure-similarity ranking; overlap -> 1 means collapse.

In [13]:
def topN_set(score_dict, src_set, N=20):
    order = [n for n, _ in sorted(((n, score_dict[n]) for n in NODE_IDS if n not in src_set), key=lambda kv: -kv[1])]
    return set(order[:N])

print("w_sem | mean top-20 overlap with pure-similarity ranking (->1 = RAG collapse)")
for w in W_SEMS:
    ovs = []
    for kind, qid, groups, man, dist in QUERIES:
        if kind != "primary":
            continue
        src_set, sim, S = scores(groups, w, 0.0)
        transport_top = topN_set(S, src_set)
        sim_top = topN_set(sim, src_set)
        ov = len(transport_top & sim_top) / 20.0
        ovs.append(ov)
    print(f"{w:5.2f} | {sum(ovs)/len(ovs):.2f}")

w_sem | mean top-20 overlap with pure-similarity ranking (->1 = RAG collapse)
 0.00 | 0.10
 0.10 | 0.30
 0.30 | 0.52
 1.00 | 0.53


## Inspector: did a SemanticProximity edge do the bridging?

For the missing-path probe (Ppass), show top-k under G0b (no edges) vs G3 (edges),
and whether the target's inflow used a semantic edge.

In [14]:
def topk(groups, w_sem, lam, k=12):
    src_set, sim, S = scores(groups, w_sem, lam)
    sc = sorted(((n, S[n]) for n in NODE_IDS if n not in src_set), key=lambda kv: -kv[1])
    for n, s in sc[:k]:
        tag = ""
        print(f"    {s:9.3f}  sim={sim[n]:.2f}  {G.nodes[n].content!r}{tag}")

g_pass = [["check in", "checking in"], ["document"]]
print("=== Ppass  target~passport/boarding pass ===")
print("--- G0b (no edges, no bypass) ---"); topk(g_pass, 0.0, 0.0)
print("--- G3 (edges w_sem=1.0, no bypass) ---"); topk(g_pass, 1.0, 0.0)

# which anchors' semantic neighbors include the target?
for a in ("check in", "document"):
    ia = resolve(a)
    if ia is None:
        continue
    nbrs = [(G.nodes[nb].content, round(sw, 2)) for nb, sw in SEM_ADJ.get(ia, [])]
    print(f"semantic neighbors of {a!r}: {nbrs[:10]}")

=== Ppass  target~passport/boarding pass ===
--- G0b (no edges, no bypass) ---
      272.349  sim=0.38  'storing information'
        0.000  sim=0.08  'coffee'
        0.000  sim=0.00  'stimulant'
        0.000  sim=0.07  'caffeine'
        0.000  sim=0.08  'served hot'
        0.000  sim=0.11  'coffee shop'
        0.000  sim=0.00  'sugar'
        0.000  sim=0.00  'made from coffee beans'
        0.000  sim=0.04  'mug'
        0.000  sim=0.05  'can'
        0.000  sim=0.18  'container'
        0.000  sim=0.01  'cup'
--- G3 (edges w_sem=1.0, no bypass) ---
      230.575  sim=0.83  'checking in'
      218.000  sim=0.76  'check in counter'
      208.165  sim=0.75  'checking out'
      145.333  sim=0.51  'registration'
      142.968  sim=0.64  'front desk'
       88.946  sim=0.89  'documentor'
       88.039  sim=0.88  'photodocument'
       85.508  sim=0.86  'e document'
       84.340  sim=1.00  'redocument'
       83.813  sim=0.84  'subdocument'
       82.816  sim=0.91  'overdocument'
  

## Interpretation checklist

1. `G0b -> G1..G3`: does median rank drop (target nearer top) as edges turn on, with bypass OFF? That is the clean structural gain.
2. `G1r..G3r` vs `G0a`: do edges add on top of the 8a bypass, or just overlap it?
3. P1: for embedding-near but graph-disconnected pairs, does the partner's rank jump from `w_sem=0` to `w_sem>0`? (edges actually bridge)
4. Danger 1: generic-distractor rank must not climb with `w_sem` (no hub re-flood).
5. Danger 2: top-20 overlap with pure-sim must not -> 1 (no RAG collapse). Find the `w_sem` where bridging happens but collapse doesn't.
6. Ppass inspector: under G3, does passport/boarding pass finally appear, and is it fed by a SemanticProximity edge?